# Vietnamese Sentiment Analysis on Student Feedback - PhoBERT

Main model: `vinai/phobert-base`

Input files required on Kaggle:

- `train_processed.csv`
- `valid_processed.csv`
- `test_processed.csv`

Preferred text column: `processed_phobert`

## 1. Install and Import Libraries

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import os
import inspect
import string
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

RANDOM_STATE = 42
MODEL_NAME = "vinai/phobert-base"
OUTPUT_DIR = Path("outputs_phobert")
PROCESSED_DATA_DIR = Path("Sentiment-Analysis-Vietnamese") / "dataset" / "processed"

TEXT_COL = "processed_phobert"
FALLBACK_TEXT_COL = "sentence"
LABEL_COL = "sentiment"
NUM_LABELS = 3
MAX_LENGTH = 128

PROCESSED_FILENAMES = {
    "train": "train_processed.csv",
    "validation": "valid_processed.csv",
    "test": "test_processed.csv",
}

LABEL_NAMES = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

set_seed(RANDOM_STATE)
OUTPUT_DIR.mkdir(exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Find and Load Processed CSV Files

In [ ]:
def find_processed_data_dir():
    required = list(PROCESSED_FILENAMES.values())
    candidate_dirs = [
        PROCESSED_DATA_DIR,
        Path("."),
        Path("dataset") / "processed",
        Path("/kaggle/working"),
    ]

    for directory in candidate_dirs:
        if directory.exists() and all((directory / filename).exists() for filename in required):
            return directory

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for train_file in kaggle_input.rglob(PROCESSED_FILENAMES["train"]):
            directory = train_file.parent
            if all((directory / filename).exists() for filename in required):
                return directory

    return None


def load_split(split_name, processed_dir):
    path = processed_dir / PROCESSED_FILENAMES[split_name]
    df = pd.read_csv(path)

    if TEXT_COL not in df.columns:
        print(f"Column {TEXT_COL!r} not found in {path.name}. Falling back to {FALLBACK_TEXT_COL!r}.")
        text_col = FALLBACK_TEXT_COL
    else:
        text_col = TEXT_COL

    required = [text_col, LABEL_COL]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")

    df = df[required].copy()
    df = df.rename(columns={text_col: "text", LABEL_COL: "label"})
    df["text"] = df["text"].fillna("").astype(str).str.strip()
    df["label"] = df["label"].astype(int)
    return df


processed_dir = find_processed_data_dir()
if processed_dir is None:
    raise FileNotFoundError("Could not find train_processed.csv, valid_processed.csv, and test_processed.csv.")

print("Using processed CSV files from:", processed_dir)
print("Preferred text column:", TEXT_COL)

train_df = load_split("train", processed_dir)
valid_df = load_split("validation", processed_dir)
test_df = load_split("test", processed_dir)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

train_df.head()

## 3. Dataset Overview

In [ ]:
for name, df in [("train", train_df), ("validation", valid_df), ("test", test_df)]:
    counts = df["label"].value_counts().sort_index()
    print(f"\n{name} label distribution:")
    for label_id, count in counts.items():
        pct = count / len(df) * 100
        print(f"  {label_id} ({LABEL_NAMES[label_id]:>8}): {count:>5} ({pct:5.2f}%)")

train_df["num_words"] = train_df["text"].str.split().map(len)
train_df["num_words"].describe()

## 4. Tokenizer and Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            max_length=max_length,
            padding=False,
        )
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = SentimentDataset(train_df["text"], train_df["label"], tokenizer, MAX_LENGTH)
valid_dataset = SentimentDataset(valid_df["text"], valid_df["label"], tokenizer, MAX_LENGTH)
test_dataset = SentimentDataset(test_df["text"], test_df["label"], tokenizer, MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 5. Model and Training Setup

In [ ]:
def load_sequence_classifier(model_name_or_path, **kwargs):
    try:
        return AutoModelForSequenceClassification.from_pretrained(
            model_name_or_path,
            attn_implementation="eager",
            **kwargs,
        )
    except TypeError:
        return AutoModelForSequenceClassification.from_pretrained(
            model_name_or_path,
            **kwargs,
        )


model = load_sequence_classifier(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=LABEL_NAMES,
    label2id={name: idx for idx, name in LABEL_NAMES.items()},
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    macro_precision, macro_recall, _, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


def make_training_args():
    kwargs = {
        "output_dir": str(OUTPUT_DIR / "trainer"),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 32,
        "num_train_epochs": 3,
        "weight_decay": 0.01,
        "logging_steps": 50,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "report_to": "none",
        "seed": RANDOM_STATE,
        "fp16": torch.cuda.is_available(),
    }

    signature = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in signature.parameters:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"

    return TrainingArguments(**kwargs)


def make_trainer(model, args, train_dataset, valid_dataset, tokenizer, data_collator):
    kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "eval_dataset": valid_dataset,
        "data_collator": data_collator,
        "compute_metrics": compute_metrics,
    }

    signature = inspect.signature(Trainer.__init__)
    if "processing_class" in signature.parameters:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in signature.parameters:
        kwargs["tokenizer"] = tokenizer

    return Trainer(**kwargs)


trainer = make_trainer(
    model=model,
    args=make_training_args(),
    train_dataset=train_dataset,
    valid_dataset=valid_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

## 6. Train

In [ ]:
trainer.train()

## 7. Evaluate on Validation and Test

In [ ]:
print("Validation metrics:")
trainer.evaluate(valid_dataset)

In [ ]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = test_df["label"].to_numpy()

test_metrics = compute_metrics((predictions.predictions, y_true))
test_metrics

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=[LABEL_NAMES[i] for i in [0, 1, 2]],
    digits=4,
    zero_division=0,
))

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[LABEL_NAMES[i] for i in [0, 1, 2]],
    yticklabels=[LABEL_NAMES[i] for i in [0, 1, 2]],
)
plt.title("Confusion Matrix - PhoBERT")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "phobert_confusion_matrix.png", dpi=200)
plt.show()

## 9. Save Outputs

In [ ]:
results_df = pd.DataFrame([{"model": "PhoBERT", **test_metrics}])
results_df.to_csv(OUTPUT_DIR / "phobert_test_results.csv", index=False, encoding="utf-8-sig")

output = test_df.copy()
output["predicted_sentiment"] = y_pred
output["true_label_name"] = output["label"].map(LABEL_NAMES)
output["predicted_label_name"] = output["predicted_sentiment"].map(LABEL_NAMES)
output["is_correct"] = output["label"] == output["predicted_sentiment"]
output.to_csv(OUTPUT_DIR / "phobert_test_predictions.csv", index=False, encoding="utf-8-sig")

trainer.save_model(str(OUTPUT_DIR / "best_phobert_model"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "best_phobert_model"))

print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")
results_df

## 10. Error Analysis Examples

In [ ]:
output[output["is_correct"] == False].head(20)

## 11. Attention Heatmaps

These heatmaps visualize which tokens PhoBERT attends to from the sentence-level `<s>` token in the last Transformer layer. Use them as an attention visualization, not as a perfect causal explanation.

In [ ]:
def clean_token_for_display(token):
    token = token.replace("@@ ", "")
    token = token.replace("@@", "")
    token = token.replace("?", "")
    return token


def should_show_token(token):
    cleaned = clean_token_for_display(token).strip()
    if not cleaned:
        return False
    if all(char in string.punctuation for char in cleaned):
        return False
    return True


def get_attention_scores(text, model, tokenizer, device):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**encoded, output_attentions=True)

    if outputs.attentions is None:
        raise RuntimeError("This model did not return attentions. Reload with attn_implementation=\"eager\".")

    last_attention = outputs.attentions[-1][0]  # [heads, seq_len, seq_len]
    cls_attention = last_attention.mean(dim=0)[0].detach().cpu().numpy()

    input_ids = encoded["input_ids"][0].detach().cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    display_tokens = []
    scores = []
    for token, score in zip(tokens, cls_attention):
        if token in tokenizer.all_special_tokens:
            continue
        if not should_show_token(token):
            continue
        display_tokens.append(clean_token_for_display(token))
        scores.append(float(score))

    scores = np.asarray(scores, dtype=np.float32)
    if len(scores) > 0 and scores.max() > scores.min():
        scores = (scores - scores.min()) / (scores.max() - scores.min())

    return display_tokens, scores.tolist()


def plot_attention_heatmap(text, title, output_path, model, tokenizer, device):
    tokens, scores = get_attention_scores(text, model, tokenizer, device)
    if not tokens:
        print("Skipped heatmap because no valid tokens were found:", title)
        return

    plt.figure(figsize=(max(8, len(tokens) * 0.45), 2.2))
    sns.heatmap(
        np.asarray([scores]),
        annot=np.asarray([tokens]),
        fmt="",
        cmap="YlOrRd",
        cbar=True,
        xticklabels=False,
        yticklabels=["attention"],
        linewidths=0.5,
        linecolor="white",
    )
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.show()


def pick_example(prediction_df, true_label, pred_label, correct):
    rows = prediction_df[
        (prediction_df["true_label_name"] == true_label)
        & (prediction_df["predicted_label_name"] == pred_label)
        & (prediction_df["is_correct"] == correct)
    ]
    if rows.empty:
        return None
    return rows.iloc[0]


def generate_attention_heatmaps(prediction_df, model, tokenizer):
    heatmap_dir = OUTPUT_DIR / "attention_heatmaps"
    heatmap_dir.mkdir(parents=True, exist_ok=True)

    device = next(model.parameters()).device
    cases = [
        ("correct_negative", "negative", "negative", True),
        ("correct_neutral", "neutral", "neutral", True),
        ("correct_positive", "positive", "positive", True),
        ("neutral_to_positive", "neutral", "positive", False),
        ("neutral_to_negative", "neutral", "negative", False),
    ]

    selected_rows = []
    for file_stem, true_label, pred_label, correct in cases:
        row = pick_example(prediction_df, true_label, pred_label, correct)
        if row is None:
            print("No example found for", file_stem)
            continue
        title = f"{file_stem}: true={true_label}, pred={pred_label}"
        output_path = heatmap_dir / f"{file_stem}.png"
        plot_attention_heatmap(row["text"], title, output_path, model, tokenizer, device)
        selected_rows.append({
            "case": file_stem,
            "true_label": true_label,
            "predicted_label": pred_label,
            "text": row["text"],
            "heatmap_file": str(output_path),
        })

    selected_df = pd.DataFrame(selected_rows)
    selected_df.to_csv(heatmap_dir / "selected_attention_examples.csv", index=False, encoding="utf-8-sig")
    print(f"Saved attention heatmaps to: {heatmap_dir.resolve()}")
    return selected_df


attention_model_path = OUTPUT_DIR / "best_phobert_model"
if attention_model_path.exists():
    attention_model = load_sequence_classifier(str(attention_model_path)).to(trainer.model.device)
else:
    attention_model = trainer.model

selected_attention_examples = generate_attention_heatmaps(output, attention_model, tokenizer)
selected_attention_examples
